# Inspect FastViT feature maps

Notebook này chạy `inspect_feature_maps.py` cho một ảnh và một checkpoint. Mặc định `scope=key` sẽ xuất activation của backbone C2–C5, FPN P2–P6, classification head, regression head, heatmap theo lớp và prediction cuối.

> Chỉ nạp checkpoint đáng tin cậy. Script dùng `torch.load(..., weights_only=False)` để đọc đầy đủ metadata training. Không chạy notebook này đồng thời với training trên cùng GPU.

## 1. Cấu hình

Nếu để đường dẫn là `None`, notebook sẽ tự tìm repository, `last.pth` và ảnh validation đầu tiên. Khi Kaggle có nhiều checkpoint cùng tên, hãy điền đường dẫn chính xác để tránh chọn nhầm.

In [ ]:
from pathlib import Path
from datetime import datetime

# Đường dẫn: có thể dùng chuỗi Windows dạng r'D:\\ml-fastvit\\...'.
REPO_ROOT_OVERRIDE = None
IMAGE_PATH = None
CHECKPOINT_PATH = None
OUTPUT_ROOT = None

# Tự clone khi chạy trên Kaggle mà chưa attach source code.
AUTO_CLONE_REPO = True
REPO_URL = 'https://github.com/chiennhm/fast_vit_opt.git'
REPO_REVISION = None  # Có thể đặt commit/tag cụ thể.
INSTALL_MISSING_DEPENDENCIES = True

# Tự tìm checkpoint theo tên. Nên đặt CHECKPOINT_PATH nếu có nhiều file.
CHECKPOINT_NAME = 'last.pth'
CHECKPOINT_SHA256 = None  # Điền SHA-256 nếu muốn bắt buộc xác minh file.

# Chế độ ghi feature map.
SCOPE = 'key'             # 'key' hoặc 'all'
INCLUDE_PATTERNS = []     # Ví dụ: [r'^fpn\.', r'^head\.']; sẽ override SCOPE
TOP_CHANNELS = 16
STATS_ONLY = False        # True: chỉ ghi thống kê, không xuất PNG activation
SAVE_TENSORS = False      # Có thể tốn rất nhiều dung lượng

DEVICE = 'auto'           # 'auto', 'cuda', 'cpu'
USE_AMP = True
IMG_SIZE = None           # None: đọc short side từ metadata checkpoint
MAX_SIZE = None           # None: đọc long side max từ metadata checkpoint
PADDING_MULTIPLE = 32
SCORE_THRESHOLD = 0.05
NMS_THRESHOLD = 0.5
PRE_NMS_TOPK = 2000
MAX_DETECTIONS = 100

# Chỉ bật cho checkpoint cũ thật sự không có metadata.architecture.
ALLOW_MISSING_CHECKPOINT_METADATA = False
MODEL = 'fastvit_sa12'
ARCHITECTURE_VERSION = 'fixed_c5'
FPN_CHANNELS = 256
ANCHOR_SIZES = [11, 19, 28, 51, 153]
REG_MAX = None

CREATE_ZIP = True
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

assert SCOPE in {'key', 'all'}
assert DEVICE in {'auto', 'cuda', 'cpu'}
assert TOP_CHANNELS > 0
assert 0 <= SCORE_THRESHOLD <= 1 and 0 <= NMS_THRESHOLD <= 1
print('RUN_ID:', RUN_ID)

## 2. Chuẩn bị repository và dependency

Trên Kaggle, nếu source nằm trong `/kaggle/input` (read-only), notebook tạo một bản source gọn trong `/kaggle/working`. Checkpoint và dataset không bị sao chép.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys

IS_KAGGLE = Path('/kaggle').is_dir()
KAGGLE_INPUT_ROOT = Path('/kaggle/input')

def run_checked(command, **kwargs):
    print('$', ' '.join(map(str, command)))
    return subprocess.run([str(x) for x in command], check=True, **kwargs)

def is_feature_debug_repo(path):
    path = Path(path)
    return (
        (path / 'inspect_feature_maps.py').is_file()
        and (path / 'detection' / 'feature_debug.py').is_file()
        and (path / 'detection' / 'fastvit_detector.py').is_file()
    )

def find_named_files_limited(root, filename, max_depth=6):
    root = Path(root)
    if not root.is_dir():
        return []
    matches = []
    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(root).parts)
        except ValueError:
            continue
        if filename in files:
            matches.append(current_path / filename)
        if depth >= max_depth:
            dirs[:] = []
    return matches

def discover_repo():
    candidates = []
    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    if IS_KAGGLE:
        candidates.extend([Path('/kaggle/working/ml-fastvit'), Path('/kaggle/working/ml-fastvit-feature-debug')])
        candidates.extend(path.parent for path in find_named_files_limited(KAGGLE_INPUT_ROOT, 'inspect_feature_maps.py'))
    for candidate in dict.fromkeys(candidates):
        if is_feature_debug_repo(candidate):
            return candidate.resolve()
    return None

REPO_ROOT = discover_repo()

# Kaggle input là read-only: chỉ sao chép source, bỏ checkpoint/dataset/artifact lớn.
if IS_KAGGLE and REPO_ROOT is not None:
    try:
        REPO_ROOT.relative_to(KAGGLE_INPUT_ROOT)
        source_in_kaggle_input = True
    except ValueError:
        source_in_kaggle_input = False
    if source_in_kaggle_input:
        writable_repo = Path('/kaggle/working/ml-fastvit-feature-debug')
        if not writable_repo.exists():
            ignore = shutil.ignore_patterns(
                '.git', '__pycache__', 'checkpoints', 'experiments', 'results',
                'runs', 'data', 'research', '*.pth', '*.pt', '*.zip'
            )
            shutil.copytree(REPO_ROOT, writable_repo, ignore=ignore)
        if not is_feature_debug_repo(writable_repo):
            raise RuntimeError(f'Bản source Kaggle không đầy đủ: {writable_repo}')
        REPO_ROOT = writable_repo.resolve()

if REPO_ROOT is None and IS_KAGGLE and AUTO_CLONE_REPO:
    clone_target = Path('/kaggle/working/ml-fastvit-feature-debug')
    if clone_target.exists() and any(clone_target.iterdir()):
        raise RuntimeError(f'{clone_target} đã tồn tại nhưng thiếu source feature debug; hãy restart session hoặc đặt REPO_ROOT_OVERRIDE.')
    run_checked(['git', 'clone', REPO_URL, clone_target])
    if REPO_REVISION:
        run_checked(['git', '-C', clone_target, 'checkout', '--detach', REPO_REVISION])
    REPO_ROOT = clone_target.resolve()

if REPO_ROOT is None or not is_feature_debug_repo(REPO_ROOT):
    raise FileNotFoundError(
        'Không tìm thấy inspect_feature_maps.py và detection/feature_debug.py. '
        'Hãy attach source project vào Kaggle, commit/push hai file này, hoặc đặt REPO_ROOT_OVERRIDE.'
    )

# Sửa tên hiển thị class ở revision cũ; tensor/class ID của checkpoint không đổi.
dataset_file = REPO_ROOT / 'bdd100k_dataset.py'
dataset_source = dataset_file.read_text(encoding='utf-8')
old_classes = '''BDD100K_CLASSES = [
    "bike",
    "bus",
    "car",
    "motor",
    "person",
    "rider",
    "traffic light",
    "traffic sign",
    "train",
    "truck",
]'''
canonical_classes = '''BDD100K_CLASSES = [
    "person",
    "rider",
    "car",
    "truck",
    "bus",
    "train",
    "motor",
    "bike",
    "traffic light",
    "traffic sign",
]'''
if old_classes in dataset_source:
    dataset_file.write_text(dataset_source.replace(old_classes, canonical_classes, 1), encoding='utf-8')
elif canonical_classes not in dataset_source:
    raise RuntimeError('Không nhận diện được BDD100K_CLASSES trong source.')

for module_name in ('torch', 'torchvision'):
    if importlib.util.find_spec(module_name) is None:
        raise ImportError(f'Thiếu {module_name}. Hãy cài PyTorch phù hợp CUDA trước khi chạy notebook.')

light_dependencies = {'timm': 'timm>=0.9.0', 'numpy': 'numpy', 'PIL': 'Pillow'}
missing = [package for module, package in light_dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING_DEPENDENCIES:
        raise ImportError(f'Thiếu dependency: {missing}')
    run_checked([sys.executable, '-m', 'pip', 'install', '-q', *missing])

print('Kaggle:', IS_KAGGLE)
print('REPO_ROOT:', REPO_ROOT)

## 3. Tìm và kiểm tra ảnh/checkpoint

Nếu cell chọn sai file, quay lại cell cấu hình và đặt `IMAGE_PATH`, `CHECKPOINT_PATH` bằng đường dẫn tuyệt đối.

In [ ]:
import hashlib
from PIL import Image
from IPython.display import display

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def find_first_validation_image(roots, max_depth=9):
    extensions = {'.jpg', '.jpeg', '.png'}
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for current, dirs, files in os.walk(root):
            current_path = Path(current)
            try:
                depth = len(current_path.relative_to(root).parts)
            except ValueError:
                continue
            parts = {part.lower() for part in current_path.parts}
            if 'val' in parts:
                candidates = sorted(name for name in files if Path(name).suffix.lower() in extensions)
                if candidates:
                    return current_path / candidates[0]
            if depth >= max_depth:
                dirs[:] = []
    return None

if CHECKPOINT_PATH is not None:
    checkpoint_path = Path(CHECKPOINT_PATH).expanduser()
else:
    checkpoint_candidates = [
        REPO_ROOT / 'checkpoints' / 'fastvit_sa12_20261008_162912' / CHECKPOINT_NAME,
        Path.cwd() / 'checkpoints' / 'fastvit_sa12_20261008_162912' / CHECKPOINT_NAME,
    ]
    if IS_KAGGLE:
        checkpoint_candidates.extend(find_named_files_limited(KAGGLE_INPUT_ROOT, CHECKPOINT_NAME, max_depth=7))
    checkpoint_candidates = [path.resolve() for path in dict.fromkeys(checkpoint_candidates) if path.is_file()]
    if not checkpoint_candidates:
        raise FileNotFoundError(f'Không tìm thấy {CHECKPOINT_NAME}; hãy đặt CHECKPOINT_PATH.')
    if len(checkpoint_candidates) > 1:
        print('Tìm thấy nhiều checkpoint; đang chọn file đầu tiên. Nên đặt CHECKPOINT_PATH để chắc chắn:')
        for candidate in checkpoint_candidates:
            print(' -', candidate)
    checkpoint_path = checkpoint_candidates[0]

if IMAGE_PATH is not None:
    image_path = Path(IMAGE_PATH).expanduser()
else:
    image_roots = [
        REPO_ROOT / 'data',
        Path('/mnt/bdd100k'),
        Path('/kaggle/working/bdd100k'),
    ]
    if IS_KAGGLE:
        image_roots.insert(0, KAGGLE_INPUT_ROOT)
    image_path = find_first_validation_image(image_roots)
    if image_path is None:
        raise FileNotFoundError('Không tìm thấy ảnh validation; hãy đặt IMAGE_PATH.')

checkpoint_path = checkpoint_path.resolve()
image_path = image_path.resolve()
if not checkpoint_path.is_file():
    raise FileNotFoundError(checkpoint_path)
if not image_path.is_file():
    raise FileNotFoundError(image_path)

actual_sha256 = sha256_file(checkpoint_path)
if CHECKPOINT_SHA256 and actual_sha256.lower() != CHECKPOINT_SHA256.lower():
    raise ValueError(f'SHA-256 checkpoint không khớp: {actual_sha256}')

if OUTPUT_ROOT is None:
    output_root = Path('/kaggle/working/feature_debug') if IS_KAGGLE else REPO_ROOT / 'runs' / 'feature_debug'
else:
    output_root = Path(OUTPUT_ROOT).expanduser()
output_root.mkdir(parents=True, exist_ok=True)

base_name = f'{image_path.stem}_{checkpoint_path.stem}_{SCOPE}_{RUN_ID}'
output_dir = output_root / base_name
suffix = 2
while output_dir.exists():
    output_dir = output_root / f'{base_name}_{suffix:02d}'
    suffix += 1

with Image.open(image_path) as preview:
    print('IMAGE:', image_path, '| size:', preview.size)
    display(preview.copy())
print('CHECKPOINT:', checkpoint_path)
print('CHECKPOINT SIZE:', f'{checkpoint_path.stat().st_size / 1024**2:.1f} MiB')
print('SHA-256:', actual_sha256)
print('OUTPUT:', output_dir)

## 4. Chạy `inspect_feature_maps.py`

`scope=all` có thể tạo hàng trăm ảnh và dùng nhiều thời gian/dung lượng. Nên chạy `scope=key`, hoặc `stats-only`, trước.

In [ ]:
import shlex

command = [
    sys.executable,
    str(REPO_ROOT / 'inspect_feature_maps.py'),
    '--image', str(image_path),
    '--checkpoint', str(checkpoint_path),
    '--output-dir', str(output_dir),
    '--scope', SCOPE,
    '--top-channels', str(TOP_CHANNELS),
    '--device', DEVICE,
    '--padding-multiple', str(PADDING_MULTIPLE),
    '--score-thresh', str(SCORE_THRESHOLD),
    '--nms-thresh', str(NMS_THRESHOLD),
    '--pre-nms-topk', str(PRE_NMS_TOPK),
    '--max-detections', str(MAX_DETECTIONS),
    '--amp' if USE_AMP else '--no-amp',
]
if STATS_ONLY:
    command.append('--stats-only')
if SAVE_TENSORS:
    command.append('--save-tensors')
for pattern in INCLUDE_PATTERNS:
    command.extend(['--include', pattern])
if IMG_SIZE is not None:
    command.extend(['--img-size', str(IMG_SIZE)])
if MAX_SIZE is not None:
    command.extend(['--max-size', str(MAX_SIZE)])
if ALLOW_MISSING_CHECKPOINT_METADATA:
    command.extend([
        '--allow-missing-checkpoint-metadata',
        '--model', MODEL,
        '--architecture-version', ARCHITECTURE_VERSION,
        '--fpn-channels', str(FPN_CHANNELS),
        '--anchor-sizes', *map(str, ANCHOR_SIZES),
    ])
    if REG_MAX is not None:
        command.extend(['--reg-max', str(REG_MAX)])

print('Command:')
print(shlex.join(command))
run_checked(command, cwd=REPO_ROOT)

## 5. Xem nhanh kết quả

In [ ]:
import json
from IPython.display import FileLink

summary_path = output_dir / 'summary.json'
if not summary_path.is_file():
    raise FileNotFoundError(f'Script chưa tạo summary: {summary_path}')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
metadata = summary['metadata']
flagged = [row for row in summary['layers'] if row.get('flags')]

print('Prediction count:', metadata.get('prediction_count'))
print('Hooked layers:', len(metadata.get('hooked_layers') or []))
print('Recorded tensors:', len(summary['layers']))
print('Class maps:', len(summary['class_maps']))
print('Flagged tensors:', len(flagged))
for row in flagged[:30]:
    print(f" - {row['layer']} call={row['call']} shape={row['shape']} flags={row['flags']}")
if len(flagged) > 30:
    print(f' ... còn {len(flagged) - 30} tensor có cờ trong index.html')

prediction_image = output_dir / 'predictions.png'
if prediction_image.is_file():
    with Image.open(prediction_image) as image:
        display(image.copy())

print('Báo cáo HTML:')
display(FileLink(str(output_dir / 'index.html')))
print('Layer CSV:')
display(FileLink(str(output_dir / 'layer_stats.csv')))
print('Class CSV:')
display(FileLink(str(output_dir / 'class_scores.csv')))

In [ ]:
if CREATE_ZIP:
    archive_path = Path(shutil.make_archive(str(output_dir), 'zip', root_dir=output_dir))
    print('ZIP:', archive_path, f'({archive_path.stat().st_size / 1024**2:.1f} MiB)')
    display(FileLink(str(archive_path)))
else:
    print('CREATE_ZIP=False; bỏ qua đóng gói output.')

## Gợi ý chạy sâu hơn

- Chỉ lấy FPN/head: `INCLUDE_PATTERNS = [r'^fpn\.', r'^head\.']`.
- Tìm layer bất thường ít tốn ổ đĩa: `SCOPE='all'`, `STATS_ONLY=True`.
- Sau khi biết layer cần xem, quay lại `SCOPE='key'` hoặc dùng `INCLUDE_PATTERNS` và đặt `STATS_ONLY=False`.
- Chỉ bật `SAVE_TENSORS=True` khi cần phân tích tensor gốc; file `.pt` có thể rất lớn.
- Mỗi lần chạy notebook tự chọn output mới, không ghi đè kết quả cũ.